# Analysis notebooks (Design 9.1 to 9.4, 9.7; figures and tables of Design 13)

This notebook only calls `scripts/analysis.py`. The functions are in `src/stats.py`.
Runtime: **CPU** is enough. Cells 0 to 5 run on placeholder data (made-up answers and
labels on the real probe ids, houses and fields) and write only to Drive `placeholder/`.
Cells 6, 7 and 8 are switched off until their time (each has a flag set to False).

In [ ]:
# Cell 0: settings, Drive, packages, and one line per input (PASS or FAIL).
# Clone or pull the repo first, the same way as in the other notebooks (classic token in Colab Secrets).
REPO = "/content/banglaindoorvqa-main"
DRIVE = "/content/drive/MyDrive/BanglaIndoorVQA"
PH = DRIVE + "/placeholder"          # placeholder files live only here (D230)

import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
assert os.path.isdir(REPO + "/.git"), "Clone the repo to " + REPO + " first"
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--format=repo HEAD %h %cI"],
                     capture_output=True, text=True).stdout)
!pip install -q sacrebleu statsmodels
%cd $REPO
!python scripts/analysis.py check-inputs --mode placeholder --inputs $PH/inputs

In [ ]:
# Cell 1: self test on invented data (needs no Drive input).
!python scripts/analysis.py selftest

In [ ]:
# Cell 2: template_id fold (D236). Counts are printed; the merged groups (question text)
# go to Drive only. One native speaker reads template_merges.csv once, before Gate E.
!python scripts/analysis.py templates --out $DRIVE/results/stats_inputs/templates

In [ ]:
# Cell 3: placeholder inputs in the real formats (scored records, HT-3, HT-4, HT-5,
# lexicon sensitivity rows, token counts). Made-up answers and labels only.
!python scripts/analysis.py placeholder --out $PH/inputs

In [ ]:
# Cell 4: the whole analysis on the placeholder inputs (10,000 resamples, seed 42).
!python scripts/analysis.py run --mode placeholder --inputs $PH/inputs --out $PH/outputs

In [ ]:
# Cell 5: list the newest placeholder run and show its five figures.
import glob
from IPython.display import Image, display
run = sorted(glob.glob(PH + "/outputs/run_*_PLACEHOLDER"))[-1]
print(run)
for f in sorted(glob.glob(run + "/**/*", recursive=True)):
    if os.path.isfile(f):
        print("%9d  %s" % (os.path.getsize(f), f[len(run) + 1:]))
for f in sorted(glob.glob(run + "/figures/*.png")):
    display(Image(filename=f, width=900))

In [ ]:
# Cell 6: question token counts for Design 9.4 item 8 (D238). Real question text, no model answer.
# Qwen and Gemma tokenizers at the pinned commits (CPU), Gemini countTokens (not billed).
# Paid tier only (Design 5.5): look at the AI Studio project first. Run once.
RUN_TOKENS = False
if RUN_TOKENS:
    !pip install -q transformers==5.18.0
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    !python scripts/analysis.py tokens --out $DRIVE/results/stats_inputs/token_counts.csv --paid-tier-confirmed

In [ ]:
# Cell 7: the HT-5 sampling frame (after Gate F, before HT-5). Error-budget category per model
# and item, ids only, from the scored files alone. The HT-5 chat draws its 140 C answers from it.
RUN_BUDGET_ITEMS = False
SCORED = DRIVE + "/scored/main/*_scored.jsonl"            # set by the Scoring chat
if RUN_BUDGET_ITEMS:
    !python scripts/analysis.py budget-items --mode real --scored "$SCORED" --out $DRIVE/results/stats_inputs/error_budget_items.csv


In [ ]:
# Cell 8: the real run. Only after Gate F, HT-4, HT-5 and asset12 `score --final`.
# The same code; only the input paths change. Output to Drive results/stats/ (never placeholder/).
RUN_REAL = False
SCORED = DRIVE + "/scored/main/*_scored.jsonl"            # set by the Scoring chat
HT3_DIR = DRIVE + "/human/HT-3/score_final_YYYYMMDDTHHMMSSZ"  # the folder of `score --final`
REAL = ('--scored "%s" --ht3-dir "%s" --ht4 "%s" --ht5 "%s" --sensitivity "%s" --tokens "%s"'
        % (SCORED, HT3_DIR, DRIVE + "/human/HT-4/ht4_labels.csv", DRIVE + "/human/HT-5/ht5_audit.csv",
           DRIVE + "/human/HT-5/lexicon_sensitivity.csv", DRIVE + "/results/stats_inputs/token_counts.csv"))
if RUN_REAL:
    !python scripts/analysis.py check-inputs --mode real $REAL
    !python scripts/analysis.py run --mode real $REAL --out $DRIVE/results/stats